<a href="https://colab.research.google.com/github/M1ztick/SAIGE/blob/main/local-trainer/SAIGE_DPO_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SAIGE — DPO Fine-Tuning (v2)
**Model**: Qwen/Qwen2.5-3B-Instruct  
**Method**: Direct Preference Optimization (DPO) via TRL  
**Dataset**: M1ztyk/SAIGE-right-speech-dpo (85 pairs, prompt-diversified: rs/generic/none conditions)  
**Output**: M1ztyk/SAIGE-dpo-v2

**What changed from v1**: Dataset now stratifies system prompt conditions across three variants per record — RS prompt, generic prompt, and no system prompt. Teaches prompt-independent behavior rather than prompt-activated response.

In [1]:
# Verify GPU
import subprocess
result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
print(result.stdout.strip())

Tesla T4, 15360 MiB


In [2]:
# Install dependencies
%pip install -q \
    transformers \
    trl \
    peft \
    accelerate \
    bitsandbytes \
    datasets \
    huggingface_hub

## Authentication
Token needs `write` scope. Get one at https://huggingface.co/settings/tokens

In [3]:
from huggingface_hub import notebook_login
notebook_login()

## Load Dataset

In [4]:
from datasets import load_dataset

dataset = load_dataset(
    "M1ztyk/SAIGE-right-speech-dpo",
    data_files="dpo_pairs.jsonl",
    split="train",
)
dataset = dataset.select_columns(["prompt", "chosen", "rejected"])

split = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train: {len(train_dataset)} pairs | Eval: {len(eval_dataset)} pairs")
print(f"Example prompt keys: {[m['role'] for m in train_dataset[0]['prompt']]}")

Train: 68 pairs | Eval: 17 pairs
Example prompt keys: ['system', 'user']


## Load Model + Tokenizer (QLoRA)
4-bit NF4 quantization via bitsandbytes — fits comfortably on T4 (16GB).

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # T4 is Turing (CC 7.5) — BF16 requires Ampere+
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Configure tokenizer for DPO training length limits
# TRL >= 0.15.0 removed max_length/max_prompt_length from DPOTrainer/DPOConfig
tokenizer.model_max_length = 1024
tokenizer.truncation_side = "left"  # Truncate long prompts from the left

print(f"Model loaded. Params: {model.num_parameters() / 1e9:.2f}B")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model loaded. Params: 3.09B


## LoRA Configuration

In [6]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

## DPO Training

Key hyperparameters — same as v1 (change data, not config, for a clean before/after):
- `beta=0.1` — how far the policy can drift from the reference model
- `lr=5e-7` — DPO is sensitive to LR; lower than SFT
- `epochs=3` — watch eval loss for overfitting at this dataset size
- `ref_model=None` — TRL uses the frozen base (pre-LoRA) as reference automatically

**Note for TRL >= 0.15.0**: `max_length` and `max_prompt_length` were removed from DPOTrainer/DPOConfig.
Length limits are now handled via tokenizer configuration (see model loading cell above).

After training, run the 2x2 ablation in the inference notebook:
adapter+RS prompt / adapter+generic prompt / base+RS prompt / base+generic prompt.
The goal: adapter+generic should look close to adapter+RS.

In [14]:
from trl import DPOTrainer, DPOConfig
from peft import get_peft_model, prepare_model_for_kbit_training, PeftModel
import inspect
import torch

OUTPUT_DIR = "./saige-dpo-v2-output"

# 1. Prepare model for quantized training
model = prepare_model_for_kbit_training(model)

# 2. Ensure LoRA is applied correctly once
if not isinstance(model, PeftModel):
    model = get_peft_model(model, lora_config)

# 3. Aggressive cast to Float16 for all trainable parameters
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float16)

# Auto-detect available parameters
dpo_trainer_params = set(inspect.signature(DPOTrainer.__init__).parameters.keys())
dpo_config_params = set(inspect.signature(DPOConfig.__init__).parameters.keys())

# Build DPOConfig
config_kwargs = dict(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-7,
    beta=0.1,
    fp16=True,
    bf16=False,
    optim="paged_adamw_32bit",
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    report_to="none",
    remove_unused_columns=False
)

if "max_length" in dpo_config_params:
    config_kwargs["max_length"] = 1024
if "max_prompt_length" in dpo_config_params:
    config_kwargs["max_prompt_length"] = 512

training_args = DPOConfig(**config_kwargs)

trainer_kwargs = dict(
    model=model,
    ref_model=None,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

# Handle tokenizer naming convention in different TRL versions
if "processing_class" in dpo_trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = DPOTrainer(**trainer_kwargs)

# Workaround for the T4 BFloat16 GradScaler error: disable scaling
if trainer.accelerator.scaler is not None:
    trainer.accelerator.scaler._enabled = False

trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
1,0.693494,0.688650,1.729784,56427.000000,-0.827425,-0.881344,0.409327,0.005956,-0.002591,0.611111,0.008547,-851.313375,-774.276408
2,0.680657,0.686820,1.730203,112854.000000,-0.826727,-0.880920,0.409292,0.016439,0.003681,0.777778,0.012758,-851.208550,-774.213684
3,0.674441,0.683294,1.730269,169281.000000,-0.826821,-0.880808,0.409481,0.024396,0.004723,0.888889,0.019672,-851.128988,-774.203274


TrainOutput(global_step=15, training_loss=0.682863998413086, metrics={'train_runtime': 557.5407, 'train_samples_per_second': 0.366, 'train_steps_per_second': 0.027, 'total_flos': 3378166279962624.0, 'train_loss': 0.682863998413086, 'epoch': 3.0})

## Push to Hub

In [ ]:
OUTPUT_REPO = "M1ztyk/SAIGE-dpo-v2"

trainer.push_to_hub(OUTPUT_REPO)
tokenizer.push_to_hub(OUTPUT_REPO)

print(f"Done. Model at: https://huggingface.co/{OUTPUT_REPO}")